# 03 — Motors and Rigid Body Motion

This notebook introduces **motors** — the Projective Geometric Algebra representation of rigid body transformations. A motor combines translation and rotation into a single even-grade element, enabling elegant kinematics computations.

## Learning Objectives

- Understand motors as the PGA representation of rigid motions
- Construct translators (pure translation)
- Construct rotors (pure rotation)
- Compose translators and rotors into motors
- Apply motors using sandwich conjugation
- Visualize trajectory evolution

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np

from amsa import Algebra

alg = Algebra.pga2d()

## 3.1 What is a Motor?

A **motor** is an even-grade element in PGA that represents a rigid body transformation:

$$M = T \cdot R = \text{translator} \times \text{rotor}$$

Properties:
- **Even grade**: grades 0 (scalar) + 2 (bivector) only
- **Dual representation**: encodes both rotation center and translation direction
- **Sandwich application**: in AMSA this is implemented as `M * P * inverse(M)` and agrees with `M P \tilde{M}` for the normalized motor examples in this notebook

This is fundamentally simpler than homogeneous transformation matrices!

## 3.2 Translators (Pure Translation)

For the bivector-form points used by AMSA's motor/viz examples, a translator that moves by `(dx, dy)` is encoded as:

$$T(dx, dy) = 1 + \frac{1}{2} dy\, e_{01} - \frac{1}{2} dx\, e_{02}$$

This sign convention is inferred from AMSA's current sandwich action on bivector-form points.

In [ ]:
# Create a translator for (0.5, 0) movement in x-direction
dx, dy = 0.5, 0.0

translator = alg.multivector({
    "e": 1.0,
    "e01": 0.5 * dy,
    "e02": -0.5 * dx,
})

origin = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})
translated = translator.sandwich(origin)

print("Translator:", translator.values)
print("Origin after translation:", translated.component("e01"), translated.component("e02"))

## 3.3 Rotors (Pure Rotation)

Rotors in PGA2d are identical to VGA2d rotors — they live in the e + e12 subspace:

$$R = \cos(\theta/2) - e_{12} \sin(\theta/2)$$

This rotates around the origin in the plane.

In [ ]:
# Create a rotor for 30-degree rotation
theta = np.deg2rad(30)

rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

print("Rotor (30°):", rotor.values)

# Apply rotor to a point
point = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": 1.0})  # (1, 0)
rotated = rotor.sandwich(point)

print("\nOriginal point (1, 0):", point.component("e01"), point.component("e02"))
print("Rotated point:", rotated.component("e01"), rotated.component("e02"))

## 3.4 Composing Motors (Translation × Rotation)

A motor combines translation and rotation:

$$M = T \times R$$

The order matters: applying $R$ then $T$ vs $T$ then $R$ gives different results.

In AMSA's current examples we use `motor = translator * rotor`, so the rightmost factor acts first inside the sandwich product.

In [ ]:
# Create a motor: translate by (0.5, 0), then rotate by 20°
dx, dy = 0.5, 0.0
theta = np.deg2rad(20)

# Translator using the codebase sign convention for bivector-form points
translator = alg.multivector({
    "e": 1.0,
    "e01": 0.5 * dy,
    "e02": -0.5 * dx,
})

# Rotor
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Motor = translator * rotor
motor = translator * rotor

print("Translator:", translator.values)
print("Rotor:", rotor.values)
print("\nMotor (T × R):", motor.values)

## 3.5 Applying Motors to Points

Like rotors, motors are applied using the sandwich product. For the normalized examples in this notebook:

$$P' = M P \tilde{M}$$

This handles both translation and rotation in one step!

In [ ]:
# Apply motor to origin point
origin = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

transformed = motor.sandwich(origin)

print("Original origin (0, 0):")
print("  x =", origin.component("e01"), ", y =", origin.component("e02"))

print("\nAfter motor transform:")
print("  x =", transformed.component("e01"), ", y =", transformed.component("e02"))

## 3.6 Visualizing Motor Trajectory

Let's create a trajectory by repeatedly applying a motor — this simulates a robot moving along a path.

In [ ]:
# Create a motor for small forward + rotation steps
theta_step = np.deg2rad(10)   # 10° per step
forward_step = 0.3

# Translator (using the same pattern as working examples)
T = alg.multivector({
    "e": 1.0,
    "e01": 0.0,
    "e02": -0.5 * forward_step,
})

# Rotor
R = alg.multivector({
    "e": np.cos(theta_step / 2),
    "e12": -np.sin(theta_step / 2)
}).normalized()

# Motor
M = T * R

# Starting point
point = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

# Compute trajectory
steps = 24  # full circle
trajectory = []

for i in range(steps):
    point = M.sandwich(point)
    x = point.component("e01")
    y = point.component("e02")
    trajectory.append([x, y])

trajectory = np.array(trajectory)

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=1.5, alpha=0.7)
ax.scatter(trajectory[:, 0], trajectory[:, 1], c=range(steps), cmap='viridis', s=30, zorder=5)
ax.scatter(0, 0, c='red', s=100, marker='*', zorder=6, label='Start')

ax.set_xlim(-1, 3)
ax.set_ylim(-1, 3)
ax.set_aspect('equal')
ax.set_title('Motor Trajectory: Forward + Turn', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 3.7 Motor Inverse

For the normalized motors in this notebook, the inverse equals the reverse:

$$M^{-1} = \tilde{M}$$

This makes backward transformations trivial!

In [ ]:
# Test inverse
M_inv = motor.reverse()

# Original point
point = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": 1.0})

# Forward
forward = motor.sandwich(point)

# Backward
backward = M_inv.sandwich(forward)

print("Original:", point.component("e01"), point.component("e02"))
print("After forward:", forward.component("e01"), forward.component("e02"))
print("After backward:", backward.component("e01"), backward.component("e02"))
print("\nRestored:", np.allclose([point.component("e01"), point.component("e02")], 
                               [backward.component("e01"), backward.component("e02")], atol=1e-10))

## 3.8 Motor Verification

Let's verify our motor works correctly by testing a sequence of motor applications to generate a trajectory.

In [ ]:
# Create a motor: forward translation + rotation (from working example)
theta_step = np.deg2rad(10)
tx, ty = 0.5, 0.0

# Rotor (same as working example)
rotor = alg.multivector({
    "e": np.cos(theta_step / 2),
    "e12": -np.sin(theta_step / 2),
}).normalized()

# Translator (same sign convention as above)
translator = alg.multivector({
    "e": 1.0,
    "e01": 0.5 * ty,
    "e02": -0.5 * tx,
})

# Motor
motor = translator * rotor

# Starting point at origin
point = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

# Apply motor multiple times - this is the same pattern as the working example
trajectory = []
for i in range(5):
    point = motor.sandwich(point)
    trajectory.append([point.component("e01"), point.component("e02")])

print("Motor trajectory (first 5 steps):")
for i, (x, y) in enumerate(trajectory):
    print(f"  Step {i+1}: ({x:.3f}, {y:.3f})")

print("\nNote: The motor follows the pattern from the working PGA examples.")
print("The exact trajectory depends on the motor's translation/rotation parameters.")

In [ ]:
# Using the working example pattern for motor construction
theta = np.deg2rad(30)
tx, ty = 1.0, 0.5

# Rotor
rotor = alg.multivector({"e": np.cos(theta/2), "e12": -np.sin(theta/2)}).normalized()

# Translator using the same sign convention as above
translator = alg.multivector({"e": 1.0, "e01": 0.5 * ty, "e02": -0.5 * tx})

# Motor
M_pga = translator * rotor

# Test point at origin
point = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

# Apply motor
point_transformed = M_pga.sandwich(point)

print("Motor transformation test:")
print("  Original: (0, 0)")
print("  After motor (tx=1, ty=0.5, rot=30°):")
print("    x =", point_transformed.component("e01"))
print("    y =", point_transformed.component("e02"))

## 3.9 Summary

We covered:

- **Translators**: Pure translation in PGA2d using AMSA's bivector-form point convention
- **Rotors**: Pure rotation (e + e12 terms)
- **Motors**: Combined translation + rotation = $T \times R$
- **Sandwich application**: for these normalized examples, `sandwich()` agrees with $M P \tilde{M}$
- **Motor inverse**: for these normalized examples, the reverse equals the inverse

In the next notebook, we'll explore **bulk and weight** — the null basis decomposition essential for PGA normalization.

## Exercises

### ⭐ Easy

**3.1** Create a motor that translates by (2, 0) and rotates by 45°. Apply it to the point (1, 1). What are the resulting coordinates?

In [ ]:
# Your turn: ⭐ Exercise 3.1
# TODO: Create motor and apply to point (1,1)
raise NotImplementedError("Implement exercise 3.1")

### ⭐⭐ Medium

**3.2** Write code to compute a motor that moves from point A to point B with a given rotation angle. Apply it to several points and verify they form the expected transformed shape.

In [ ]:
# Your turn: ⭐⭐ Exercise 3.2
# Create a motor from (0,0) to (2,1) with 30° rotation
def motor_from_to(start, end, angle):
    """Create motor that translates and rotates from start to end."""
    # TODO: Implement
    raise NotImplementedError("Implement motor_from_to function")

M = motor_from_to((0, 0), (2, 1), np.pi/6)
# Test on a square of points
test_points = [(0, 0), (1, 0), (1, 1), (0, 1)]
# TODO: Transform and visualize
raise NotImplementedError("Implement exercise 3.2")

### ⭐⭐⭐ Challenge

**3.3** Create a function that interpolates between two motors: given motors M1 and M2 and parameter t ∈ [0, 1], compute the interpolated motor. This is useful for motion planning. Hint: use the exponential map (log of motors).

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 3.3
def interpolate_motors(M1, M2, t):
    """Interpolate between two motors."""
    # TODO: Use log/exp for smooth interpolation
    raise NotImplementedError("Implement interpolate_motors function")

# Create two motors
T1 = alg.multivector({"e": 1.0, "e01": 0.0})
R1 = alg.multivector({"e": np.cos(np.pi/8), "e12": -np.sin(np.pi/8)}).normalized()
M1 = T1 * R1

T2 = alg.multivector({"e": 1.0, "e01": 2.0})
R2 = alg.multivector({"e": np.cos(np.pi/4), "e12": -np.sin(np.pi/4)}).normalized()
M2 = T2 * R2

# Interpolate at t=0.5
M_mid = interpolate_motors(M1, M2, 0.5)
print("Mid-motor:", M_mid.values)

## Attribution

This notebook draws on:

- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **Geometric Algebra for Computer Graphics** — John Vince
  https://link.springer.com/book/10.1007/978-1-84628-997-2
- **PGABLE Tutorial** — Leger and Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf